# Z Image Turbo · Atelier GGUF Engine

This version preserves the optimization strategy from the supplied notebook: **ComfyUI + ComfyUI-GGUF**, automatic GGUF/UNet routing, and model-only LoRA loading. It does not use the full Diffusers pipeline.

The launch cell starts a local ComfyUI API and exposes a custom Gradio studio with a share link. The default low-VRAM checkpoint is `z_image_turbo-Q4_K_M.gguf`; you can switch to Q5/Q6/Q8 or a standard `.safetensors` UNet URL when more memory is available.

Two T4 cards do not pool into 32 GB. This engine uses GPU 0 for one ComfyUI worker; the second card is not falsely counted as extra VRAM.

Metadata cleanup is privacy-only: it removes private EXIF/XMP/workflow details and does not fabricate camera data or bypass provenance/moderation systems.

In [ ]:
#@title Launch Z Image Turbo Atelier · ComfyUI GGUF
# One cell: installs ComfyUI, ComfyUI-GGUF, launches the API, then opens a custom Gradio studio.

import os, sys, subprocess, time, json, re, io, zipfile, threading, urllib.request, urllib.parse, shutil, gc
from pathlib import Path

def sh(cmd): subprocess.check_call(cmd, shell=True)
sh("pip -q install -U gradio requests pillow pandas huggingface_hub")
BASE = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
COMFY = BASE/'ComfyUI'; OUT = COMFY/'output'; MODELS=COMFY/'models'; LORAS=MODELS/'loras'
for p in [COMFY,OUT,MODELS,LORAS]: p.mkdir(parents=True,exist_ok=True)
if not (COMFY/'main.py').exists(): sh(f"git clone -q https://github.com/comfyanonymous/ComfyUI {COMFY}")
sh(f"pip -q install -r {COMFY/'requirements.txt'}")
GGUF_NODE=COMFY/'custom_nodes/ComfyUI-GGUF'
if not GGUF_NODE.exists(): sh(f"mkdir -p {GGUF_NODE.parent} && git clone -q https://github.com/city96/ComfyUI-GGUF {GGUF_NODE}")
if (GGUF_NODE/'requirements.txt').exists(): sh(f"pip -q install -r {GGUF_NODE/'requirements.txt'}")

import requests, pandas as pd, gradio as gr
from PIL import Image

TEXT_URL='https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors'
VAE_URL='https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors'
DEFAULT_GGUF='https://huggingface.co/jayn7/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q4_K_M.gguf'
COMFY_URL='http://127.0.0.1:8188'
SERVER=None
STOP=threading.Event(); RUN=threading.Lock()
ASPECTS={'Instagram square · 1:1':(640,640,1080,1080),'Instagram portrait · 4:5':(640,800,1080,1350),'Instagram story/reel · 9:16':(576,1024,1080,1920),'Facebook landscape · 1.91:1':(768,416,1200,630),'YouTube thumbnail · 16:9':(768,432,1280,720),'Pinterest portrait · 2:3':(640,960,1000,1500),'Native 1024 · 1:1':(1024,1024,1024,1024)}
BLOCK=[r'\b(child|kid|minor|underage|preteen|toddler|baby|infant)\b',r'\b(young[- ]looking|schoolgirl|schoolboy|teen\s*(girl|boy)?|barely legal)\b',r'\b(non[- ]consensual|without consent|revenge porn|upskirt|hidden camera)\b',r'\b(real person|celebrity|public figure)\b.*\b(nude|nsfw|sex|explicit)\b']

def gpu_text():
    try:
        import torch
        return ' | '.join(f'GPU {i}: {torch.cuda.get_device_name(i)} · {torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GiB' for i in range(torch.cuda.device_count())) or 'No CUDA GPU'
    except Exception: return 'GPU status unavailable'

def safe_reason(p):
    for pat in BLOCK:
        if re.search(pat,(p or '').lower()): return 'Blocked by safety guard.'
    return ''

def fetch(url, folder):
    if not url or not url.strip(): return None
    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)
    name=urllib.parse.unquote(Path(urllib.parse.urlparse(url).path).name) or 'download.bin'
    if '?' in name: name=name.split('?')[0]
    dest=folder/name
    if dest.exists() and dest.stat().st_size>0: return dest
    with requests.get(url.strip(),stream=True,timeout=60) as r:
        r.raise_for_status()
        with open(dest,'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk: f.write(chunk)
    return dest

def start_comfy():
    global SERVER
    try: requests.get(COMFY_URL,timeout=2); return
    except Exception: pass
    SERVER=subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188'],cwd=COMFY,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    for _ in range(90):
        try: requests.get(COMFY_URL,timeout=2); return
        except Exception: time.sleep(2)
    raise RuntimeError('ComfyUI did not start. Check the cell output and GPU runtime.')

def list_files(folder, exts): return sorted([p.name for p in Path(folder).glob('*') if p.suffix.lower() in exts])

def assets(): return list_files(MODELS/'unet',{'.gguf','.safetensors','.bin'}) + list_files(MODELS/'diffusion_models',{'.gguf','.safetensors','.bin'})
def loras(): return ['None']+list_files(LORAS,{'.safetensors','.bin','.pt'})

def prepare(model_url, lora_url):
    # Default components match the supplied notebook; model can be GGUF or standard safetensors.
    fetch(TEXT_URL,MODELS/'clip'); fetch(VAE_URL,MODELS/'vae')
    model=fetch(model_url,MODELS/'unet')
    if lora_url and lora_url.strip(): fetch(lora_url,LORAS)
    start_comfy()
    return gr.update(choices=assets(),value=model.name if model else (assets()[0] if assets() else None)), gr.update(choices=loras()), f'Assets ready · {gpu_text()}'

def workflow(model, prompt, negative, width, height, seed, lora, lora_strength):
    w={
      '9':{'inputs':{'filename_prefix':'atelier/z-image','images':['43',0]},'class_type':'SaveImage'},
      '39':{'inputs':{'clip_name':'qwen_3_4b.safetensors','type':'lumina2','device':'default'},'class_type':'CLIPLoader'},
      '40':{'inputs':{'vae_name':'ae.safetensors'},'class_type':'VAELoader'},
      '41':{'inputs':{'width':int(width),'height':int(height),'batch_size':1},'class_type':'EmptySD3LatentImage'},
      '42':{'inputs':{'text':negative or 'blurry, low quality, deformed, artifacts','clip':['39',0]},'class_type':'CLIPTextEncode'},
      '43':{'inputs':{'samples':['44',0],'vae':['40',0]},'class_type':'VAEDecode'},
      '44':{'inputs':{'seed':int(seed),'steps':8,'cfg':0.0,'sampler_name':'res_multistep','scheduler':'beta','denoise':1,'model':['47',0],'positive':['45',0],'negative':['42',0],'latent_image':['41',0]},'class_type':'KSampler'},
      '45':{'inputs':{'text':prompt,'clip':['39',0]},'class_type':'CLIPTextEncode'},
      '47':{'inputs':{'shift':3.0,'model':['48',0]},'class_type':'ModelSamplingAuraFlow'},
    }
    if str(model).lower().endswith('.gguf'): w['48']={'inputs':{'unet_name':model},'class_type':'UnetLoaderGGUF'}
    else: w['48']={'inputs':{'unet_name':model,'weight_dtype':'default'},'class_type':'UNETLoader'}
    if lora and lora!='None':
        w['50']={'inputs':{'lora_name':lora,'strength_model':float(lora_strength),'model':['48',0]},'class_type':'LoraLoaderModelOnly'}
        w['47']['inputs']['model']=['50',0]
    return w

def queue(w):
    r=requests.post(COMFY_URL+'/prompt',json={'prompt':w},timeout=60); r.raise_for_status(); return r.json()['prompt_id']
def wait_image(pid):
    while not STOP.is_set():
        r=requests.get(COMFY_URL+f'/history/{pid}',timeout=30).json()
        if pid in r:
            for o in r[pid].get('outputs',{}).values():
                for im in o.get('images',[]): return OUT/im['filename']
        time.sleep(.7)
    try: requests.post(COMFY_URL+'/interrupt',timeout=5)
    except Exception: pass
    return None

def clean_copy(src,dest,fmt):
    im=Image.open(src).convert('RGB')
    # Pixel re-encode intentionally omits EXIF/XMP/workflow metadata.
    im.save(dest,'JPEG' if fmt=='JPEG' else 'PNG',quality=95 if fmt=='JPEG' else None,exif=b'') if fmt=='JPEG' else im.save(dest,'PNG',optimize=True)
    return im

def bulk_prompts(single,bulk):
    if bulk: return [x.strip() for x in Path(bulk).read_text(encoding='utf-8',errors='replace').splitlines() if x.strip()]
    return [single.strip()] if single and single.strip() else []

def generate(model,prompt,negative,bulk,aspect,count,seed,lora,lora_strength,fmt,upscale,progress=gr.Progress()):
    if not RUN.acquire(False): raise gr.Error('A run is already active.')
    STOP.clear(); gallery=[]; rows=[]; n=0
    try:
        if not model: raise gr.Error('Prepare assets first.')
        prompts=bulk_prompts(prompt,bulk)
        if not prompts: raise gr.Error('Enter a prompt or upload a .txt file.')
        gw,gh,ow,oh=ASPECTS[aspect]; start_comfy()
        for pi,p in enumerate(prompts):
            if safe_reason(p): rows.append({'status':'blocked','prompt':p}); continue
            for j in range(int(count)):
                if STOP.is_set():
                    yield gallery,'Stopped via ComfyUI interrupt. Completed files remain available.',None; return
                n+=1; progress((pi+(j+.1)/max(1,int(count)))/len(prompts),desc=f'Queuing image {n}')
                pid=queue(workflow(model,p,negative,gw,gh,int(seed)+n-1,lora,lora_strength)); src=wait_image(pid)
                if src is None: yield gallery,'Stopped.',None; return
                im=Image.open(src).convert('RGB')
                if upscale and (im.width,im.height)!=(ow,oh): im=im.resize((ow,oh),Image.Resampling.LANCZOS)
                slug=re.sub(r'[^a-zA-Z0-9]+','_',p).strip('_')[:50] or 'prompt'; ext='jpg' if fmt=='JPEG' else 'png'; dest=BASE/'atelier_exports'; dest.mkdir(exist_ok=True); out=dest/f'{n:04d}_{slug}.{ext}'; clean_copy(src,out,fmt); gallery.append((im,f'{n:04d} · seed {int(seed)+n-1}')); rows.append({'status':'ok','file':out.name,'prompt':p,'seed':int(seed)+n-1,'width':im.width,'height':im.height}); yield gallery,f'Generated {n} image(s) · {gpu_text()}',None
        manifest=BASE/'atelier_exports'/'manifest.csv'; pd.DataFrame(rows).to_csv(manifest,index=False); z=BASE/'Z_Image_Turbo_Atelier_GGUF.zip'
        with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as zz:
            for f in (BASE/'atelier_exports').iterdir(): zz.write(f,'images/'+f.name)
        yield gallery,f'Complete · {n} image(s)',str(z)
    finally: STOP.clear(); RUN.release()

def stop():
    STOP.set()
    try: requests.post(COMFY_URL+'/interrupt',timeout=5)
    except Exception: pass
    return 'Stop requested — ComfyUI interrupt sent.'

def stage_lora(upload):
    if not upload: return gr.update(choices=loras()),'No LoRA selected.'
    p=Path(upload); dest=LORAS/p.name; shutil.copy2(p,dest); return gr.update(choices=loras(),value=p.name),f'Staged {p.name}'

CSS=r"""
body,.gradio-container{background:radial-gradient(1200px 700px at 75% -10%,#3b2a55,#11101a 52%,#09090e)!important;color:#f3efe8!important}.gradio-container{max-width:1500px!important}#mast{padding:38px 44px 28px;border:1px solid #695b7c88;border-radius:28px;background:#171420d9;box-shadow:0 24px 80px #000a;margin-bottom:18px}#mast h1{font:500 clamp(38px,6vw,78px)/.9 Georgia,serif;letter-spacing:-.05em;margin:0;color:#fffaf2}#mast em{color:#d7ff64;font-style:normal}#mast p{font:14px/1.6 ui-monospace,monospace;color:#c8bfd6;max-width:800px;margin:18px 0 0}.panel{border:1px solid #63567177!important;border-radius:22px!important;background:#15131fbe!important;box-shadow:0 18px 55px #05050988!important}textarea,input,select{background:#0c0b12!important;border-color:#51485e!important;color:#f7f2eb!important}button{border-radius:12px!important}#go button{background:#d7ff64!important;color:#15131d!important;font-weight:700!important}#halt button{background:#2b1e31!important;color:#ffcabd!important}#archive button{background:#1d2431!important;color:#d8e7ff!important}footer{display:none!important}
"""

start_comfy()
with gr.Blocks(css=CSS,theme=gr.themes.Base(),title='Z Image Turbo Atelier GGUF') as app:
    gr.HTML("<div id='mast'><div style='color:#d7ff64;letter-spacing:.18em;font:11px ui-monospace,monospace'>COMFYUI / GGUF / Z IMAGE TURBO</div><h1>Make the image<br><em>feel inevitable.</em></h1><p>The same low-VRAM engine as the supplied notebook: quantized GGUF routing, model-only LoRA loading, and a visual studio for serious bulk runs.</p></div>")
    with gr.Row():
        with gr.Column(scale=5,elem_classes='panel'):
            gr.Markdown('### Direction')
            prompt=gr.Textbox(label='Single prompt',lines=5,placeholder='A moonlit editorial portrait, oxidized silver, wet pavement…')
            bulk=gr.File(label='Bulk prompts · one prompt per line',file_types=['.txt'],type='filepath')
            negative=gr.Textbox(value='blurry, low quality, deformed, artifacts',label='Negative prompt')
            with gr.Row(): aspect=gr.Dropdown(list(ASPECTS),value='Instagram square · 1:1',label='Aspect'); count=gr.Slider(1,50,1,step=1,label='Images / prompt')
            with gr.Row(): seed=gr.Number(12345,precision=0,label='Base seed'); fmt=gr.Radio(['PNG','JPEG'],value='PNG',label='Export')
            upscale=gr.Checkbox(False,label='Upscale clean pixels to social preset dimensions')
        with gr.Column(scale=4,elem_classes='panel'):
            gr.Markdown('### Engine / LoRA')
            model_url=gr.Textbox(value=DEFAULT_GGUF,label='GGUF or safetensors model URL')
            model=gr.Dropdown(assets(),label='Installed model')
            lora=gr.Dropdown(loras(),value='None',label='LoRA file'); strength=gr.Slider(0,2,1,step=.05,label='LoRA strength')
            lora_upload=gr.File(label='Stage LoRA',file_types=['.safetensors','.bin','.pt'],type='filepath')
            lora_url=gr.Textbox(label='Optional LoRA URL'); stage=gr.Button('Stage local LoRA'); prepare_btn=gr.Button('Prepare assets / refresh engine')
            info=gr.Markdown(gpu_text())
    with gr.Row():
        go=gr.Button('Generate run',elem_id='go'); halt=gr.Button('Stop immediately',elem_id='halt'); archive=gr.Button('Build / download ZIP',elem_id='archive')
    status=gr.Markdown('Ready. Default model: Q4_K_M GGUF.'); gallery=gr.Gallery(columns=4,height='auto',label='Completed images'); download=gr.File(label='ZIP export')
    prepare_btn.click(prepare,[model_url,lora_url],[model,lora,info])
    stage.click(stage_lora,lora_upload,[lora,status])
    go.click(generate,[model,prompt,negative,bulk,aspect,count,seed,lora,strength,fmt,upscale],[gallery,status,download])
    halt.click(stop,outputs=status)
    archive.click(lambda: str(BASE/'Z_Image_Turbo_Atelier_GGUF.zip') if (BASE/'Z_Image_Turbo_Atelier_GGUF.zip').exists() else None,outputs=download)
app.queue(default_concurrency_limit=1).launch(share=True,show_error=True)